In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Intel only
# !pip install scikit-learn-intelex
from sklearnex import patch_sklearn
patch_sklearn()

Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [3]:
import sys 
import os
sys.path.append('..')
import gc

In [4]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import time
import re

In [5]:
UPSAMPLE_LATER = True 
UPSAMPLE_RATIO = 1

RETRAIN_KNN = True
RETRAIN_TRANSFORM = True

USING_SMOTE = False
FILL_MEAN = True

USING_CACHE = False

In [6]:
from utils.check_feature import power_scaler_col

In [7]:
appl_train = pd.read_csv('../data/dseb63_application_train.csv', index_col=0)

# ====== BUREAU ======

bureau = pd.read_parquet('../data/dseb63_bureau_general_v3.parquet')
bureau_columns = bureau.columns

new_bureau_columns = {col: 'BUREAU_' + col for col in bureau_columns if col != 'SK_ID_CURR'} 
bureau.rename(columns=new_bureau_columns, inplace=True)

# ====== PREVIOUS APPLICATION ======

prev_app = pd.read_parquet('../data/prev_app_hanh_v1_1.parquet')
prev_app_columns = prev_app.columns

new_prev_app_columns = {col: 'PREV_APP_' + col for col in prev_app_columns if col != 'SK_ID_CURR'}
prev_app.rename(columns=new_prev_app_columns, inplace=True)

# ====== INSTALLMENTS PAYMENTS ======

installments = pd.read_parquet('../data/dseb63_installment_gb.parquet')
# installments = pd.read_csv('../data/dseb63_installment_gb.csv')
installments_columns = installments.columns

installments.drop(columns= [col for col in installments_columns if 'TARGET' in col], inplace=True)

new_installments_columns = {col: 'INSTALLMENTS_' + col for col in installments_columns if col != 'SK_ID_CURR'}
installments.rename(columns=new_installments_columns, inplace=True)

# ====== CREDIT CARD BALANCE ======

credit_card = pd.read_parquet('../data/dseb63_credit_card_balance_gb_v1_1.parquet')
# credit_card = pd.read_parquet('../data/dseb63_credit_card_balance_gb_v2.parquet')
credit_card_columns = credit_card.columns

new_credit_card_columns = {col: 'CREDIT_CARD_' + col for col in credit_card_columns if col != 'SK_ID_CURR'}
credit_card.rename(columns=new_credit_card_columns, inplace=True)

# ====== POS CASH BALANCE ======

pos_cash = pd.read_parquet('../data/dseb63_pos_cash_gb_v2.parquet') # Good
# pos_cash = pd.read_csv('../data/dseb63_pos_cash_gb.csv')
pos_cash_columns = pos_cash.columns

new_posh_cash_columns = {col: 'POS_CASH_' + col for col in pos_cash_columns if col != 'SK_ID_CURR'}
pos_cash.rename(columns=new_posh_cash_columns, inplace=True)

# ====== MERGE APP PREV APP ======

app_prev_app = pd.read_parquet('../data/dseb63_app_prev_app_features.parquet')
app_prev_app_columns = app_prev_app.columns

new_app_prev_app_columns = {col: 'APP_PREV_APP_' + col for col in app_prev_app_columns if col != 'SK_ID_CURR'}
app_prev_app.rename(columns=new_app_prev_app_columns, inplace=True)

# ====== MERGE DATA ======

df = appl_train.merge(bureau, on='SK_ID_CURR', how='left')
df = df.merge(installments, on='SK_ID_CURR', how='left')
df = df.merge(pos_cash, on='SK_ID_CURR', how='left')
df = df.merge(credit_card, on='SK_ID_CURR', how='left')
df = df.merge(prev_app, on='SK_ID_CURR', how='left')


# df = df.merge(app_prev_app, on='SK_ID_CURR', how='left')


In [8]:
def RELU(series):
    return series.apply(lambda x: max(0, x))
    

In [9]:
# del bureau, installments, pos_cash, credit_card, prev_app
# gc.collect()

In [10]:
appl_train['WALLSMATERIAL_MODE'].value_counts()

WALLSMATERIAL_MODE
Panel           52873
Stone, brick    51662
Block            7399
Wooden           4314
Mixed            1851
Monolithic       1432
Others           1298
Name: count, dtype: int64

In [11]:
def process_df(app_train):
    # df = df.copy()
    app_train['CODE_GENDER'].replace('XNA',np.nan, inplace=True)
    app_train['CNT_CHILDREN'] = app_train['CNT_CHILDREN'].replace([8,9,10,11,14,19],7)
    app_train['sin_HOUR_APPR_PROCESS_START'] = np.sin(2 * np.pi * app_train['HOUR_APPR_PROCESS_START'] / 24)
    app_train['cos_HOUR_APPR_PROCESS_START'] = np.cos(2 * np.pi * app_train['HOUR_APPR_PROCESS_START'] / 24)
    app_train['NAME_FAMILY_STATUS'].replace('Unknown', 'Single / not married', inplace=True)

    app_train['EMERGENCYSTATE_MODE'].fillna('No', inplace=True)
    app_train['FONDKAPREMONT_MODE'].fillna('Unknown', inplace=True)
    app_train['WALLSMATERIAL_MODE'].fillna('Not Specified', inplace=True) 
    app_train['OCCUPATION_TYPE'].replace('IT staff', 'High skill tech staff', inplace=True)
    app_train['OCCUPATION_TYPE'].replace('Realty agents', 'Sales staff', inplace=True)
    app_train['OCCUPATION_TYPE'].replace('HR staff', 'Laborers', inplace=True)
    app_train['OCCUPATION_TYPE'].fillna('Unknown', inplace=True)
    app_train['OCCUPATION_TYPE'].replace(['Cleaning staff', 'Cooking staff', 'Waiters/barmen staff'], 'F&B staff', inplace=True)
    app_train['HOUSETYPE_MODE'].replace(['block of flats', 'specific housing'], 'house', inplace=True)
    app_train['WALLSMATERIAL_MODE'].replace(['Others', 'Mixed', 'Monolithic'], 'Others', inplace=True)
    app_train['WALLSMATERIAL_MODE'].replace(['Block', 'Stone, brick'], 'Brick', inplace=True)

    def group_organization_type(org_type):
        if 'Trade' in org_type:
            return 'Trade'
        elif 'Industry' in org_type:
            return 'Industry'
        elif 'Business' in org_type:
            return 'Business Entity'
        elif 'Transport' in org_type:
            return 'Transport'
        else:
            return org_type
    app_train['ORGANIZATION_TYPE'] = app_train['ORGANIZATION_TYPE'].replace('XNA', 'Unknown')
    app_train['ORGANIZATION_TYPE'] = app_train['ORGANIZATION_TYPE'].apply(group_organization_type)
    
    #app_train = app_train[app_train['OWN_CAR_AGE'] != 91.0]
    app_train['OWN_CAR_AGE'] = app_train['OWN_CAR_AGE'].fillna(-100)
    #app_train = app_train[app_train['FLAG_MOBIL'] == 1]

    app_train['EXT_SOURCE_SUM'] = app_train[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].sum(axis=1, skipna=True)
    app_train['EXT_SOURCE_PROD'] = app_train[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].prod(axis=1, skipna=True)
    app_train['EXT_SOURCE_MEAN'] = app_train[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1, skipna=True)
    app_train['EXT_SOURCE_STD'] = app_train[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1, skipna=True)
    app_train['EXT_SOURCE_MISSING_VALUES'] = app_train[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].isna().sum(axis=1)
    app_train['EXT_SOURCE_WEIGHTED'] = app_train.EXT_SOURCE_1 * 2 + app_train.EXT_SOURCE_2 * 3 + app_train.EXT_SOURCE_3 * 4
    
    app_train['AMT_REQ_CREDIT_BUREAU_YEAR'] = app_train['AMT_REQ_CREDIT_BUREAU_YEAR'].fillna(value = app_train['AMT_REQ_CREDIT_BUREAU_YEAR'].mean())

    #Documents 
    docs = [f for f in app_train.columns if 'FLAG_DOC' in f]
    app_train['NEW_DOC_IND_AVG'] = app_train[docs].mean(axis=1)
    app_train['NEW_DOC_IND_STD'] = app_train[docs].std(axis=1)
    app_train['DOCUMENT_COUNT'] = app_train[docs].sum(axis=1)
    app_train['NEW_DOC_IND_KURT'] = app_train[docs].kurtosis(axis=1)
    app_train['HAS_DOCUMENT'] = app_train[docs].max(axis=1)
    #Drop flag document  
    app_train = app_train.drop(app_train.filter(regex='^FLAG_DOCUMENT_\d+$').columns, axis = 1)

    #New feat from income and inquiry 
    app_train['TOTALAREA_MODE'] = app_train['TOTALAREA_MODE'] ** (1/3)
    med_income = app_train.groupby(['ORGANIZATION_TYPE', 'NAME_EDUCATION_TYPE'])['AMT_INCOME_TOTAL'].transform('median')
    med_income2 = app_train.groupby('ORGANIZATION_TYPE')['AMT_INCOME_TOTAL'].transform('median')
    app_train['income_ratio'] = app_train['AMT_INCOME_TOTAL'] / med_income
    app_train['income_ratio2'] = app_train['AMT_INCOME_TOTAL'] / med_income2
    app_train['true_annuity_div_income'] = app_train['AMT_ANNUITY'] / med_income
    app_train['true_annuity_div_income2'] = app_train['AMT_ANNUITY'] / med_income2
    app_train['true_income_div_totalarea'] = med_income / app_train['TOTALAREA_MODE'].clip(0.001,1)
    app_train['true_income_div_totalarea2'] = med_income2 / app_train['TOTALAREA_MODE'].clip(0.001,1)

    #New features from bureau 
    app_train['TOTAL_ENQUIRIES_CREDIT_BUREAU'] = app_train[['AMT_REQ_CREDIT_BUREAU_DAY',
                                            'AMT_REQ_CREDIT_BUREAU_HOUR',
                                            'AMT_REQ_CREDIT_BUREAU_WEEK',
                                            'AMT_REQ_CREDIT_BUREAU_MON',
                                            'AMT_REQ_CREDIT_BUREAU_QRT',
                                            'AMT_REQ_CREDIT_BUREAU_YEAR']].sum(axis=1)

    app_train['PCTG_ENQUIRIES_HOUR'] = app_train['AMT_REQ_CREDIT_BUREAU_HOUR'] / app_train['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    app_train['PCTG_ENQUIRIES_DAY'] = app_train['AMT_REQ_CREDIT_BUREAU_DAY'] / app_train['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    app_train['PCTG_ENQUIRIES_WEEK'] = app_train['AMT_REQ_CREDIT_BUREAU_WEEK'] / app_train['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    app_train['PCTG_ENQUIRIES_MON'] = app_train['AMT_REQ_CREDIT_BUREAU_MON'] / app_train['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    app_train['PCTG_ENQUIRIES_QRT'] = app_train['AMT_REQ_CREDIT_BUREAU_QRT'] / app_train['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    app_train['PCTG_ENQUIRIES_YEAR'] = app_train['AMT_REQ_CREDIT_BUREAU_YEAR'] / app_train['TOTAL_ENQUIRIES_CREDIT_BUREAU']
    
    #Adding new features; percentages
    app_train['DAYS_EMPLOYED_PERC'] = app_train['DAYS_EMPLOYED'] / app_train['DAYS_BIRTH']
    app_train['INCOME_CREDIT_PERC'] = app_train['AMT_INCOME_TOTAL'] / app_train['AMT_CREDIT']
    app_train['INCOME_PER_PERSON'] = app_train['AMT_INCOME_TOTAL'] / app_train['CNT_FAM_MEMBERS']
    app_train['ANNUITY_INCOME_PERC'] = app_train['AMT_ANNUITY'] / app_train['AMT_INCOME_TOTAL']
    app_train['PAYMENT_RATE'] = app_train['AMT_ANNUITY'] /app_train['AMT_CREDIT']
    app_train['DEBT_BURDEN_PER_WORKING_DAY'] = app_train['PAYMENT_RATE'] / (app_train['DAYS_EMPLOYED'] + 1e-5) * (app_train['DAYS_EMPLOYED'] != 0)
    app_train['DEBT_BURDEN_PER_LIFE_DAY'] = app_train['PAYMENT_RATE'] / app_train['DAYS_BIRTH']
    # ratio of the difference between the loan amount and the value of goods compared to the value of the goods
    app_train['CREDIT_GOODS_PRICE_RATIO1'] = (app_train['AMT_CREDIT'] - app_train['AMT_GOODS_PRICE']) /  app_train['AMT_GOODS_PRICE']
    # ratio of the difference between the loan amount and the value of goods compared to the loan amount
    app_train['CREDIT_GOODS_PRICE_RATIO2'] = (app_train['AMT_CREDIT'] - app_train['AMT_GOODS_PRICE']) /  app_train['AMT_CREDIT']
    app_train['DIFF_OBS_30_CNT_SOCIAL_CIRCLE_OBS_60_CNT_SOCIAL_CIRCLE'] = app_train['OBS_30_CNT_SOCIAL_CIRCLE'] - app_train['OBS_60_CNT_SOCIAL_CIRCLE']
    app_train['DIFF_DEF_30_CNT_SOCIAL_CIRCLE_DEF_60_CNT_SOCIAL_CIRCLE'] = app_train['DEF_30_CNT_SOCIAL_CIRCLE'] - app_train['DEF_60_CNT_SOCIAL_CIRCLE']
    
    app_train["HOUR_APPR_PROCESS_START"] = app_train["HOUR_APPR_PROCESS_START"].replace([8,9,10,11,12,13,14,15,16,17], 'working_hours')
    app_train["HOUR_APPR_PROCESS_START"] = app_train["HOUR_APPR_PROCESS_START"].replace([18,19,20,21,22,23,0,1,2,3,4,5,6,7], 'off_hours')
    app_train.drop(columns=['HOUR_APPR_PROCESS_START'], inplace=True)

    missing_columns = ['APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG',
                'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG',
                'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG',
                'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG']
    app_train['MISSING_GRADINGS'] = app_train[missing_columns].isna().sum(axis=1)
    map_week_day = {
    'MONDAY': 'week_day',
    'TUESDAY': 'week_day',
    'WEDNESDAY': 'week_day',
    'THURSDAY': 'week_day',
    'FRIDAY': 'week_day',
    'SATURDAY': 'weekend',
    'SUNDAY': 'weekend',
    }
    app_train['WEEKDAY_APPR_PROCESS_START'] = app_train['WEEKDAY_APPR_PROCESS_START'].map(map_week_day)
    map_edu = {
        'Lower secondary': 0,
        'Secondary / secondary special': 1,
        'Incomplete higher': 2,
        'Higher education': 3,
        'Academic degree': 5
    }
    app_train['NAME_EDUCATION_TYPE'] = app_train['NAME_EDUCATION_TYPE'].map(map_edu, na_action='ignore').astype(int).fillna(0)

    #app_train['NAME_FAMILY_STATUS'].replace('Unknown', 'Single / not married', inplace=True)
    #df['CODE_GENDER'].replace('XNA', 'F', inplace=True)
    app_train['DAYS_EMPLOYED'] = app_train['DAYS_EMPLOYED'].fillna(365243)

    others = app_train['NAME_INCOME_TYPE'].value_counts().index[4:]
    app_train['NAME_INCOME_TYPE'].replace(others, 'Others', inplace=True)
    app_train['NAME_TYPE_SUITE'].fillna('Unaccompanied', inplace=True)
    app_train['TOTALAREA_MODE'].fillna(0, inplace=True)
    app_train['AGE_INT'] = -df['DAYS_BIRTH'] // 365
    app_train['DAYS_LAST_PHONE_CHANGE'].replace(0, np.nan, inplace=True)
    app_train['car_to_birth_ratio'] = RELU(app_train['OWN_CAR_AGE'] / app_train['DAYS_BIRTH'])
    app_train['car_to_employ_ratio'] = RELU(app_train['OWN_CAR_AGE'] / app_train['DAYS_EMPLOYED'])
    app_train['children_ratio'] = app_train['CNT_CHILDREN'] / app_train['CNT_FAM_MEMBERS']
    app_train['credit_to_annuity_ratio'] = app_train['AMT_CREDIT'] / app_train['AMT_ANNUITY'] 
    app_train['credit_to_goods_ratio'] = app_train['AMT_CREDIT'] / app_train['AMT_GOODS_PRICE']
    app_train['credit_to_income_ratio'] = app_train['AMT_CREDIT'] / app_train['AMT_INCOME_TOTAL']   

    app_train['income_per_child'] = app_train['AMT_INCOME_TOTAL'] / (1 + app_train['CNT_CHILDREN'])
    app_train['phone_to_birth_ratio'] = RELU(app_train['DAYS_LAST_PHONE_CHANGE'] / app_train['DAYS_BIRTH'])
    app_train['phone_to_employ_ratio'] = RELU(app_train['DAYS_LAST_PHONE_CHANGE'] / (app_train['DAYS_EMPLOYED'] + 1e-5)) * (app_train['DAYS_EMPLOYED'] != 0) 
    app_train['cnt_non_child'] = app_train['CNT_FAM_MEMBERS'] - app_train['CNT_CHILDREN']
    app_train['child_to_non_child_ratio'] = app_train['CNT_CHILDREN'] / app_train['cnt_non_child'] * (app_train['cnt_non_child'] > 0)
    app_train['income_per_non_child'] = app_train['AMT_INCOME_TOTAL'] / app_train['cnt_non_child']* (app_train['cnt_non_child'] > 0)
    app_train['credit_per_person'] = app_train['AMT_CREDIT'] / app_train['CNT_FAM_MEMBERS']* (app_train['cnt_non_child'] > 0)
    app_train['credit_per_child'] = app_train['AMT_CREDIT'] / (1 + app_train['CNT_CHILDREN'])
    app_train['credit_per_non_child'] = app_train['AMT_CREDIT'] / app_train['cnt_non_child']* (app_train['cnt_non_child'] > 0)
    app_train['CREDIT_DOWN_PAYMENT'] = app_train['AMT_GOODS_PRICE'] - app_train['AMT_CREDIT']
    
    app_train['LIVINGAREA_AVG'] = app_train['LIVINGAREA_AVG'].fillna(0)
    app_train['LANDAREA_AVG'] = app_train['LANDAREA_AVG'].fillna(0)
    app_train['FLOORSMAX_AVG'] = app_train['FLOORSMAX_AVG'].fillna(0)
    app_train['LIVINGAPARTMENTS_AVG'] = app_train['LIVINGAPARTMENTS_AVG'].fillna(0)
    app_train['YEARS_BUILD_AVG'] = app_train['YEARS_BUILD_AVG'].fillna(0)

    # some random numerical transformation
    app_train['LIVINGAREA_AVG'] = app_train['LIVINGAREA_AVG'] ** (1/3.5)
    app_train['NONLIVINGAPARTMENTS_AVG'] = app_train['NONLIVINGAPARTMENTS_AVG'] ** (1/7)
    app_train['NONLIVINGAREA_AVG'] = app_train['NONLIVINGAREA_AVG'] ** (1/5)
    app_train['OBS_30_CNT_SOCIAL_CIRCLE'] = app_train['OBS_30_CNT_SOCIAL_CIRCLE'] ** (1/7)
    app_train['DEF_30_CNT_SOCIAL_CIRCLE'] = app_train['DEF_30_CNT_SOCIAL_CIRCLE'] ** (1/7)
    app_train['OBS_60_CNT_SOCIAL_CIRCLE'] = app_train['OBS_60_CNT_SOCIAL_CIRCLE'] ** (1/7)
    app_train['DEF_60_CNT_SOCIAL_CIRCLE'] = app_train['DEF_60_CNT_SOCIAL_CIRCLE'] ** (1/7)
   
    app_train['REGION_POPULATION_RELATIVE'] = np.sqrt(app_train['REGION_POPULATION_RELATIVE'])
    app_train['APARTMENTS_AVG'] = np.log1p(50 * app_train['APARTMENTS_AVG'])
    app_train['YEARS_BEGINEXPLUATATION_AVG'] = app_train['YEARS_BEGINEXPLUATATION_AVG'] ** 30
    app_train['YEARS_BUILD_AVG'] = app_train['YEARS_BUILD_AVG'] ** 3
    app_train['COMMONAREA_AVG'] = app_train['COMMONAREA_AVG'].clip(0.000001,1) ** (-1/200)
    app_train['ELEVATORS_AVG'] = app_train['ELEVATORS_AVG'] ** (1/40)
    app_train['ENTRANCES_AVG'] = app_train['ENTRANCES_AVG'] ** (1/3)
    app_train['FLOORSMAX_AVG'] = app_train['FLOORSMAX_AVG'] ** (1/2.5)
    app_train['FLOORSMIN_AVG'] = app_train['FLOORSMIN_AVG'] ** (1/2.2)
    app_train['LANDAREA_AVG'] = app_train['LANDAREA_AVG'] ** (1/5)
    app_train['LIVINGAPARTMENTS_AVG'] = app_train['LIVINGAPARTMENTS_AVG'] ** (1/3)

    #Some new ratio 
    app_train['RATIO_AMT_GOODS_PRICE_TO_LIVINGAREA_AVG'] = app_train['AMT_GOODS_PRICE'] / app_train['LIVINGAREA_AVG'].clip(0.001,1)
    app_train['RATIO_AMT_GOODS_PRICE_TO_LANDAREA_AVG'] = app_train['AMT_GOODS_PRICE'] / app_train['LANDAREA_AVG'].clip(0.001,1)
    app_train['RATIO_AMT_GOODS_PRICE_TO_FLOORSMAX_AVG_AVG'] = app_train['AMT_GOODS_PRICE'] / app_train['FLOORSMAX_AVG'].clip(0.001,1)
    app_train['RATIO_AMT_GOODS_PRICE_TO_LIVINGAPARTMENTS_AVG'] = app_train['AMT_GOODS_PRICE'] / app_train['LIVINGAPARTMENTS_AVG'].clip(0.001,1)
    app_train['RATIO_AMT_GOODS_PRICE_TO_YEARS_BUILD_AVG'] = app_train['AMT_GOODS_PRICE'] / app_train['YEARS_BUILD_AVG'].clip(0.001,1)
    app_train['RELIABILITY_IN_CUSTOMER_CITY'] = app_train['REG_CITY_NOT_LIVE_CITY'] + app_train['REG_CITY_NOT_WORK_CITY'] + app_train['REG_REGION_NOT_LIVE_REGION'] + app_train['REG_REGION_NOT_WORK_REGION'] + app_train['LIVE_CITY_NOT_WORK_CITY'] + app_train['LIVE_REGION_NOT_WORK_REGION']
    app_train['SUM_CONTACTS'] = app_train['FLAG_MOBIL'] + app_train['FLAG_EMP_PHONE'] + app_train['FLAG_WORK_PHONE'] + app_train['FLAG_CONT_MOBILE'] + app_train['FLAG_PHONE'] + app_train['FLAG_EMAIL']


    
    
    return app_train


In [12]:
    # df['RATIO_AMT_GOODS_PRICE_TO_LIVINGAREA_AVG'] = (df['AMT_GOODS_PRICE'] / df['LIVINGAREA_AVG'].clip(0.05,1) +1e-5) * (df['LIVINGAREA_AVG'] / df['LIVINGAREA_AVG']+1e-5)
    # df['RATIO_AMT_GOODS_PRICE_TO_LANDAREA_AVG'] = (df['AMT_GOODS_PRICE'] / df['LANDAREA_AVG'].clip(0.05,1) +1e-5) * (df['LANDAREA_AVG'] / df['LANDAREA_AVG']+1e-5)
    # df['RATIO_AMT_GOODS_PRICE_TO_FLOORSMAX_AVG_AVG'] = (df['AMT_GOODS_PRICE'] / df['FLOORSMAX_AVG'].clip(0.05,1) +1e-5) * (df['FLOORSMAX_AVG'] / df['FLOORSMAX_AVG']+1e-5)
    # df['RATIO_AMT_GOODS_PRICE_TO_LIVINGAPARTMENTS_AVG'] = (df['AMT_GOODS_PRICE'] / df['LIVINGAPARTMENTS_AVG'].clip(0.05,1) +1e-5) * (df['LIVINGAPARTMENTS_AVG'] / df['LIVINGAPARTMENTS_AVG']+1e-5)
    # df['RATIO_AMT_GOODS_PRICE_TO_YEARS_BUILD_AVG'] = (df['AMT_GOODS_PRICE'] / df['YEARS_BUILD_AVG'].clip(0.05,1) +1e-5) * (df['YEARS_BUILD_AVG'] / df['YEARS_BUILD_AVG']+1e-5)
    
    
    # df['REGION_POPULATION_RELATIVE'] = np.sqrt(df['REGION_POPULATION_RELATIVE'])
    # df['APARTMENTS_AVG'] = np.log1p(50 * df['APARTMENTS_AVG'])
    # df['YEARS_BUILD_AVG'] = df['YEARS_BUILD_AVG'] ** 3
    # df['COMMONAREA_AVG'] = df['COMMONAREA_AVG'].clip(0.0001,1) ** (-1/5)
    # df['ELEVATORS_AVG'] = df['ELEVATORS_AVG'] ** (1/10)
    # df['ENTRANCES_AVG'] = df['ENTRANCES_AVG'] ** (1/3)
    # df['FLOORSMAX_AVG'] = df['FLOORSMAX_AVG'] ** (1/2.5)
    # df['FLOORSMIN_AVG'] = df['FLOORSMIN_AVG'] ** (1/2.2)
    # df['LANDAREA_AVG'] = df['LANDAREA_AVG'] ** (1/5)
    # df['LIVINGAPARTMENTS_AVG'] = df['LIVINGAPARTMENTS_AVG'] ** (1/3)
    # df['LIVINGAREA_AVG'] = df['LIVINGAREA_AVG'] ** (1/3)
    # df['NONLIVINGAPARTMENTS_AVG'] = df['NONLIVINGAPARTMENTS_AVG'] ** (1/5)
    # df['NONLIVINGAREA_AVG'] = df['NONLIVINGAREA_AVG'] ** (1/3)
    # df['OBS_30_CNT_SOCIAL_CIRCLE'] = df['OBS_30_CNT_SOCIAL_CIRCLE'] ** (1/5)
    # df['DEF_30_CNT_SOCIAL_CIRCLE'] = df['DEF_30_CNT_SOCIAL_CIRCLE'] ** (1/5)
    # df['OBS_60_CNT_SOCIAL_CIRCLE'] = df['OBS_60_CNT_SOCIAL_CIRCLE'] ** (1/5)
    # df['DEF_60_CNT_SOCIAL_CIRCLE'] = df['DEF_60_CNT_SOCIAL_CIRCLE'] ** (1/5)

In [13]:
df = process_df(df)

In [14]:
def display_missing_data_info(dataframe, ascending=False):
    missing_values_count = dataframe.isnull().sum()
    missing_values_percentage = (dataframe.isnull().mean() * 100)
    missing_data = pd.DataFrame({
        'Missing Values': missing_values_count,
        'Percentage (%)': missing_values_percentage
    })
    
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    sorted_missing_data = missing_data.sort_values(by='Missing Values', ascending=ascending)

    print(sorted_missing_data)
    return sorted_missing_data

In [15]:
check_null_df = display_missing_data_info(df)

                              Missing Values  Percentage (%)
BUREAU_BAD_DEBT_SUM_CREDIT            245993       99.993496
BUREAU_BAD_DEBT_LAST_5_YEAR           245993       99.993496
BUREAU_BAD_DEBT_FINISHED              245993       99.993496
BUREAU_COUNT_BAD_DEBT                 245993       99.993496
BUREAU_CLOSE_LATENCY_30_DAYS          245252       99.692288
...                                      ...             ...
credit_per_person                          1        0.000406
credit_per_non_child                       1        0.000406
children_ratio                             1        0.000406
CNT_FAM_MEMBERS                            1        0.000406
income_per_non_child                       1        0.000406

[863 rows x 2 columns]


In [16]:
df.shape

(246009, 936)

In [17]:
high_null_cols = [
    'BUREAU_SOLD_SUM_CREDIT',
    'BUREAU_BAD_DEBT_LAST_5_YEAR',
    'BUREAU_BAD_DEBT_FINISHED',
    'BUREAU_BAD_DEBT_SUM_CREDIT',
    'BUREAU_CLOSE_LATENCY_30_DAYS',
    'BUREAU_SUM_OTHER',
    'BUREAU_SUM_OVERDUE_MICROLOAN',
    'BUREAU_SUM_DEBT_MICROLOAN',
    'BUREAU_SUM_MICROLOAN',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIMARY_MIN',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIVILEGED_MAX',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIVILEGED_MEAN',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIMARY_MAX',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIVILEGED_MIN',
    'PREV_APP_APPROVED_RATE_INTEREST_PRIMARY_MEAN',
    'BUREAU_SOLD_DURATION',
    'BUREAU_COUNT_SOLD',
    'BUREAU_SOLD_LAST_1000_DAYS',
    'BUREAU_SOLD_SUM_CREDIT', # 98.3

    'BUREAU_ACTIVE_SUM_CREDIT_30_DAYS', # 97.16
    'BUREAU_ACTIVE_SUM_DEBT_30_DAYS',
    'BUREAU_ACTIVE_SUM_OVERDUE_30_DAYS',

    'BUREAU_CLOSE_LATENCY_180_DAYS', # 96.6
    'BUREAU_CLOSE_SUM_DEBT_180_DAYS', # 96.54
    'BUREAU_CLOSE_SUM_OVERDUE_180_DAYS',
    'BUREAU_CLOSE_SUM_CREDIT_180_DAYS',
    # 'BUREAU_CLOSE_LATENCY_365_DAYS', #86

    'BUREAU_SUM_MORTGAGE', # 95.35 -> 0.5644
    'BUREAU_SUM_DEBT_MORTGAGE',
    'BUREAU_SUM_OVERDUE_MORTGAGE',

    'PREV_APP_REFUSED_AMT_DOWN_PAYMENT_STD', # 94.84

    'BUREAU_SUM_CREDIT_CAR_LOAN', # 93.611
    'BUREAU_SUM_OVERDUE_CAR_LOAN',
    'BUREAU_SUM_DEBT_CAR_LOAN',

    'BUREAU_CLOSE_SUM_CREDIT_LIFE_TIME', # 91.91

    'PREV_APP_REFUSED_AMT_ANNUITY_SKEW', # 91.5
    'PREV_APP_REFUSED_AMT_GOODS_PRICE_SKEW',
    'PREV_APP_REFUSED_AMT_CREDIT_SKEW', # 89.9
    'PREV_APP_REFUSED_AMT_APPLICATION_SKEW',

    'POS_CASH_LONG_TERM_LAST_36_MONTHS', # 88.5
    'POS_CASH_LONG_TERM_CNT_INSTALMENT',
    'POS_CASH_LONG_TERM_LONG_TERM',
    'POS_CASH_LONG_TERM_SK_DPD',
    'POS_CASH_LONG_TERM_SK_DPD_DEF',
    'POS_CASH_LONG_TERM_RATE_COMPLETED',
    'POS_CASH_LONG_TERM_SK_ID_PREV',
    'POS_CASH_LONG_TERM_NUM_INSTALMENT',
    'POS_CASH_LONG_TERM_LAST_12_MONTHS', # 0.56612

    'PREV_APP_REFUSED_RATE_DOWN_PAYMENT_MAX', # 85.3
    'PREV_APP_REFUSED_AMT_DOWN_PAYMENT_MAX',
    'PREV_APP_REFUSED_RATE_DOWN_PAYMENT_MEAN',
    'PREV_APP_REFUSED_RATE_DOWN_PAYMENT_MIN',
    'PREV_APP_REFUSED_AMT_DOWN_PAYMENT_MEAN',
    'PREV_APP_REFUSED_AMT_DOWN_PAYMENT_MIN',

    'PREV_APP_REFUSED_RATIO_GOODS_TO_ANNUITY_STD', # 84.7
    'PREV_APP_REFUSED_RATIO_APPLICATION_TO_ANNUITY_STD',
    'PREV_APP_REFUSED_CREDIT_ANNUITY_RATIO_STD',
    'PREV_APP_REFUSED_ANNUITY_PAYMENT_PRODUCT_STD',
    'PREV_APP_REFUSED_AMT_ANNUITY_STD',
    'PREV_APP_REFUSED_AMT_GOODS_PRICE_STD', # 0.56626

    'PREV_APP_REFUSED_APP_CREDIT_PERC_VAR', # 83.96
    'PREV_APP_REFUSED_AMT_CREDIT_STD',
    'PREV_APP_REFUSED_AMT_APPLICATION_STD',
    'CREDIT_CARD_PERCENTAGE_OF_MINIMUM_PAYMENTS_MISSED_sum', # 0.56654

    # 80

    # Add cc_balance_v2
    'CREDIT_CARD_PERCENTAGE_OF_MINIMUM_PAYMENTS_MISSED_var',
    'CREDIT_CARD_SUM_ALL_AMT_DRAWINGS_var',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_var',
    'CREDIT_CARD_SUM_ALL_CNT_DRAWINGS_var',
    'CREDIT_CARD_CNT_DRAWINGS_ATM_CURRENT_std',
    'CREDIT_CARD_PERCENTAGE_OF_MINIMUM_PAYMENTS_MISSED_max',
    'CREDIT_CARD_SUM_ALL_AMT_DRAWINGS_max',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_mean',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_max',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_min',



    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_std' 
    'CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_std',
    'CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_std',
    'CREDIT_CARD_SUM_ALL_CNT_DRAWINGS_mean',
    'CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_mean',
    'CREDIT_CARD_CNT_DRAWINGS_ATM_CURRENT_mean',
    'CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_max',
    'CREDIT_CARD_AMT_DRAWINGS_ATM_CURRENT_min',
    'CREDIT_CARD_SUM_ALL_AMT_DRAWINGS_mean',
    'CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_min',
    'CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_max',
    'CREDIT_CARD_AMT_DRAWINGS_POS_CURRENT_mean',
    'CREDIT_CARD_RATIO_ALL_AMT_DRAWINGS_TO_ALL_CNT_DRAWINGS_mean', # 0.56664

    'BUREAU_GENERAL_AMT_ANNUITY_std',
    'BUREAU_ACTIVE_SUM_OVERDUE_LIFE_TIME',
    'BUREAU_ACTIVE_SUM_CREDIT_LIFE_TIME', # 0.56670

]

In [18]:
# df.drop(columns=['PREV_APP_APPROVED_AMT_DOWN_PAYMENT_SKEW'], inplace=True)

In [19]:
ratio_keyword = ['_']
pct_cols = []
threshold_null = 90 # 75 -> 0.56656

for col in df.columns:
    for keyword in ratio_keyword:
        if col in check_null_df.index and keyword in col.lower() and check_null_df.loc[col]['Percentage (%)'] > threshold_null:
        
            pct_cols.append(col)

In [20]:
# pct_cols

In [21]:
# pct_cols.remove('BUREAU_Ratio C')
# pct_cols.remove('BUREAU_Ratio X')
# pct_cols.remove('BUREAU_Ratio 0')
# pct_cols.remove('BUREAU_Ratio Other')

In [22]:
# high_null_cols += pct_cols

In [23]:
check_null_df.to_excel('../temp/check_null.xlsx')

In [24]:
X_test_ = pd.read_csv('../data/dseb63_application_test.csv',  index_col=0)
X_test_ = X_test_.merge(bureau, on='SK_ID_CURR', how='left')
X_test_ = X_test_.merge(installments, on='SK_ID_CURR', how='left')
X_test_ = X_test_.merge(pos_cash, on='SK_ID_CURR', how='left')
X_test_ = X_test_.merge(credit_card, on='SK_ID_CURR', how='left')
X_test_ = X_test_.merge(prev_app, on='SK_ID_CURR', how='left')


# X_test_ = X_test_.merge(app_prev_app, on='SK_ID_CURR', how='left')
X_test_ = process_df(X_test_)

In [25]:
for col in high_null_cols:
    if col in X_test_.columns:
        X_test_[col].fillna(0, inplace=True)
        df[col].fillna(0,inplace= True)

In [26]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer, OrdinalEncoder, LabelEncoder, MinMaxScaler

In [27]:
# X = pd.read_parquet('../data/df_train.parquet')
X = df
del df

In [28]:
y = X['TARGET']
X.drop(columns=['TARGET', 'SK_ID_CURR'], inplace=True)

test_id = X_test_['SK_ID_CURR']
X_test_.drop(columns=['SK_ID_CURR'], inplace=True)


In [29]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246009 entries, 0 to 246008
Columns: 934 entries, NAME_CONTRACT_TYPE to SUM_CONTACTS
dtypes: float64(893), int64(26), object(15)
memory usage: 1.7+ GB


In [30]:
ordinal_cols = ['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE', 'CODE_GENDER', 'NAME_CONTRACT_TYPE']
float_cols = []
int_cols = []
cate_cols = []
flag_cols = []
for col in X.columns:
    if col not in ordinal_cols:
        if X[col].dtype == 'float64':
            float_cols.append(col)
            X[col] = X[col].astype('float64')
        elif X[col].dtype == 'int64':

            int_cols.append(col)
            X[col] = X[col].astype('int64')
        else:
            cate_cols.append(col)
            X[col] = X[col].astype('str')
    else:
        X[col] = X[col].astype('str')

Find suitable column for power transformation

In [31]:
# X[float_cols+int_cols].clip(-999999999, 999999999, inplace=True)
# X_test_[float_cols+int_cols].clip(-999999999, 999999999, inplace=True)

In [32]:
# TEST

X2 = X.copy()

for col in float_cols:
    X2[col].fillna(X2[col].mean(), inplace=True)
    
for col in int_cols:
    X2[col].fillna(X2[col].mean(), inplace=True)
for col in cate_cols:
    X2[col].fillna('Unknown', inplace=True)

In [33]:
power_col, standard_col, min_max_col = power_scaler_col(X2[float_cols+int_cols], skewness= 3, kurtosis = 20, use_cache=USING_CACHE)

100%|██████████| 919/919 [00:04<00:00, 198.29it/s]


3 / 20 Get me 566

In [34]:
len(power_col), len(standard_col), len(min_max_col)

(461, 349, 109)

In [35]:
if FILL_MEAN:
    for col in float_cols:
        X[col].fillna(X[col].mean(), inplace=True)
        X_test_[col].fillna(X[col].mean(), inplace=True)
        
    for col in int_cols:
        X[col].fillna(X[col].mean(), inplace=True)
        X_test_[col].fillna(X[col].mean(), inplace=True)
        
    for col in cate_cols:
        X[col].fillna(np.nan, inplace=True)
        X_test_[col].fillna(np.nan, inplace=True)

else:
    # for col in float_cols:
    #     X_train_[col].fillna(0, inplace=True)
    #     X_val_[col].fillna(0, inplace=True)
        
    # for col in int_cols:
    #     X_train_[col].fillna(0, inplace=True)
    #     X_val_[col].fillna(0, inplace=True)
        
    # for col in cate_cols:
    #     X_train_[col].fillna('Unknown', inplace=True)
    #     X_val_[col].fillna('Unknown', inplace=True)
    pass

In [36]:
X_columns = X.columns

### ADD KNN Feature

In [37]:
y = y.to_numpy()

In [38]:
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=500, n_jobs=-1))  
])

knn_pipeline_2 = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=500, n_jobs=-1))  
])

In [39]:
onehot_transformer = OneHotEncoder(handle_unknown='ignore')
power_transformer = PowerTransformer()
ordinal_transformer = OrdinalEncoder()
scaler_transformer = StandardScaler()
min_max_transformer = MinMaxScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_transformer, cate_cols),
        ('power', power_transformer, power_col), # power_transformer
        ('ordinal', ordinal_transformer, ordinal_cols),
        ('scale', scaler_transformer, standard_col),
        ('min_max', min_max_transformer, min_max_col)
        
    ],
    n_jobs=-1
)




In [40]:
weak_df_col = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'credit_to_annuity_ratio', 'ANNUITY_INCOME_PERC']
weak_df_col_2 = ['DAYS_ID_PUBLISH', 'DAYS_REGISTRATION', 'DAYS_EMPLOYED_PERC', 'car_to_birth_ratio', 'NAME_EDUCATION_TYPE', 'AGE_INT']

## Upsample - Downsample

### Manual double

In [41]:
#from imblearn.over_sampling import SMOTENC
from sklearn.utils import resample

In [42]:
def resample_data(X_train, y_train, rate = 0.2, decay = 0.7, return_idx = False, sampling_strategy = 0.1):
    
    posidx = (y_train == 1)
    negidx = (y_train == 0)

    # X_train = np.concatenate([X_train, y_train2], axis=1)
    posidx = np.where(posidx)[0]
    negidx = np.where(negidx)[0]

    n_samples = int(len(negidx) * rate)

    if posidx.sum() < n_samples: # Not enough positive samples
        return X_train, y_train
    
    if USING_SMOTE:
        if sampling_strategy == 'auto':
            sampling_strategy = 0.5 * (len(posidx) / len(negidx))

        knn = KNeighborsClassifier(n_neighbors=50, n_jobs=-1) # Train a weak KNN model
        knn.fit(X_train, y_train)
        y_temp = knn.predict_proba(X_train[posidx, :])[:,1] # Get the probability of being positive

        border_idx = np.where(y_temp < sampling_strategy)[0] # Find the border samples ()
        mask = np.zeros(len(y_temp), dtype=bool) 
        mask[border_idx] = True

        important_idx = posidx[mask]
        not_important_idx = posidx[~mask]

        len_import = len(important_idx)
        len_not_import = len(not_important_idx)
        len_data = len(posidx)

        reverse_decay = (len_data - len_not_import * decay) / len_import

        upscale_important = int(len_import/len_data * n_samples * reverse_decay)
        upscale_not_important = int(len_not_import/len_data * n_samples * decay)

        important_idx2 = resample(important_idx, n_samples=upscale_important, random_state=42)
        not_important_idx2 = resample(not_important_idx, n_samples=upscale_not_important, random_state=42)

        print('Near border sample:',len_import, 'Up to', upscale_important)
        print('Far border sample:',len_not_import, 'Up to', upscale_not_important)

        posidx2 = np.concatenate([important_idx2, not_important_idx2])
        

    else:
        posidx2 = resample(posidx, n_samples=n_samples, random_state=42)

    idx = np.concatenate([posidx2, negidx])
    np.random.shuffle(idx)
    
    if return_idx:
        return idx

    if isinstance(X_train, pd.DataFrame):
        return X_train.iloc[idx,:], y_train[idx]
    else:
        return X_train[idx,:], y_train[idx]

In [43]:
X_train = X.copy()

In [44]:
X_train.shape[1]

934

In [45]:
num_features = X_train.shape[1]
num_samples = X_train.shape[0]

In [46]:
!export USING_CACHE=false

In [47]:
if USING_CACHE \
    and os.path.exists(f'../temp/X_train_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy') \
    and os.path.exists(f'../temp/X_test_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy'):

    print('Load cached data')
    X_train = np.load(f'../temp/X_train_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy')
    X_test = np.load(f'../temp/X_test_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy')

    y = np.load(f'../temp/y_train_{num_samples}_{num_features}.npy')
    feature_names = np.load(f'../temp/feature_names_{num_samples}_{num_features}.npy', allow_pickle=True)

else:
    print('Fit weak features')
    start = time.time()
    knn_pipeline.fit(X_train[weak_df_col], y) # fit on original data
    knn_pipeline_2.fit(X_train[weak_df_col_2], y) # fit on original data


    weak_feature_train = knn_pipeline.predict_proba(X_train[weak_df_col])[:,1]
    weak_feature_val = knn_pipeline.predict_proba(X_test_[weak_df_col])[:,1]

    weak_feature_train_2 = knn_pipeline_2.predict_proba(X_train[weak_df_col_2])[:,1]
    weak_feature_val_2 = knn_pipeline_2.predict_proba(X_test_[weak_df_col_2])[:,1]

    end = time.time()
    print('Weak features fitted', end-start)

    print('Transforming data')
    start = time.time()
    
    preprocessor.fit(X) # fit on original data
    X_train = preprocessor.transform(X_train)
    X_test = preprocessor.transform(X_test_)
    end = time.time()
    print('Data transformed', end-start)


    X_train = np.concatenate([X_train, 
                                weak_feature_train.reshape(-1,1), 
                                weak_feature_train_2.reshape(-1,1)
                                ], axis=1)
    X_test = np.concatenate([X_test, 
                            weak_feature_val.reshape(-1,1), 
                            weak_feature_val_2.reshape(-1,1)
                            ], axis=1)

    cat_feature = preprocessor.named_transformers_['onehot'].get_feature_names_out(cate_cols)
    feature_names = np.concatenate([cat_feature, 
                                    power_col, 
                                    ordinal_cols, 
                                    standard_col, 
                                    min_max_col, 
                                    [
                                        'WEAK_FEATURE', 
                                        'WEAK_FEATURE_2',
                                    ]])

    np.save(f'../temp/X_train_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy', X_train)
    np.save(f'../temp/X_test_{num_samples}_{num_features}_{len(power_col)}_{len(standard_col)}.npy', X_test)

    np.save(f'../temp/y_train_{num_samples}_{num_features}.npy', y)
    np.save(f'../temp/feature_names_{num_samples}_{num_features}.npy', feature_names)


Fit weak features


Weak features fitted 23.252467393875122
Transforming data
Data transformed 74.93279886245728


In [48]:
idx = resample_data(X_train, y, rate = UPSAMPLE_RATIO, decay = 0.99, return_idx = True, sampling_strategy = 0.1)
X_resampled = X_train[idx,:]
y_resampled = y[idx]

In [49]:
print('Resampled data shape:', X_resampled.shape)

Resampled data shape: (452266, 1013)


In [50]:
#X_resampled = pd.DataFrame(X_resampled, columns=feature_names)
#X_test = pd.DataFrame(X_test, columns=feature_names)

#X_resampled['TARGET'] = y_resampled


#y_test = pd.read_csv('../temp/target.csv', index_col=0)
#X_test['TARGET'] = y_test['True_Target'].values

#X_resampled.to_parquet('../temp/X_train.parquet', index=False)
#X_test.to_parquet('../temp/X_val.parquet', index=False)

#X_resampled.drop(columns=['TARGET'], inplace=True)
#X_test.drop(columns=['TARGET'], inplace=True)

In [51]:
# raise Exception('Stop here')

In [52]:

from sklearn.model_selection import cross_val_score, StratifiedKFold, cross_validate
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [53]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np

CHUA CHAY BELOW

In [54]:
pca = PCA()
pca.fit(X_resampled)

PCA()

In [55]:
explained_variance = pca.explained_variance_ratio_

# Get the absolute values of PCA components (loadings)
feature_contributions = np.abs(pca.components_)

# Aggregate feature importance scores
# Weight each feature's contribution by the variance explained by the component
weighted_contributions = feature_contributions * explained_variance[:, np.newaxis]

# Sum weighted contributions across components
feature_importance = weighted_contributions.sum(axis=0)

# Create a DataFrame for ranking
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)
importance_df.reset_index(drop=True, inplace=True)

In [56]:
importance_df

,Feature,Importance
0,INSTALLMENTS_LAST_30_DAYS_sum,3.123885e-02
1,INSTALLMENTS_LAST_60_DAYS_sum,3.053979e-02
2,INSTALLMENTS_LAST_90_DAYS_sum_LONG_TERM,3.035716e-02
3,PREV_APP_PREV_CODE_REJECT_REASON_HC_SUM,2.986875e-02
4,BUREAU_SUM_DEBT_MICROLOAN,2.979487e-02
...,...,...
1008,PREV_APP_APPROVED_NAME_CONTRACT_TYPE_XNA_SUM,4.744053e-17
1009,POS_CASH_NOT_COMPLETED_SHORT_CNT_INSTALMENT,2.581396e-17
1010,POS_CASH_NOT_COMPLETED_LONG_CNT_INSTALMENT,2.445391e-17
1011,BUREAU_COUNT_BAD_DEBT,3.711695e-18


In [57]:
def select_features_with_pca(X):
    
    # Convert to DataFrame if X is a NumPy array
    if isinstance(X, np.ndarray):
        X = pd.DataFrame(X, columns=[f"Feature{i+1}" for i in range(X.shape[1])])
    
    # Apply PCA
    pca = PCA()
    pca.fit(X)
    
    # Calculate absolute loadings (contributions of features to components)
    explained_variance = pca.explained_variance_ratio_
    feature_contributions = np.abs(pca.components_)
    weighted_contributions = feature_contributions * explained_variance[:, np.newaxis]
    # Identify features exceeding the threshold for any component
    
    feature_importance = weighted_contributions.sum(axis=0)
    
    # Filter the data to keep only selected features
    # selected_data = X[feature_mask, :]
    
    return feature_importance

IF USING PCA

In [58]:
# feature_importance = select_features_with_pca(X_resampled)
# feature_map = (feature_importance > 0.0001)

In [59]:
feature_map = np.ones(len(feature_names), dtype=bool)

In [60]:
feature_map.sum()

1013

In [61]:
feature_important_df  = pd.read_excel('../temp/feature_importance_big.xlsx')
threshold_important = 250
threshold_split = 4

mask_important = feature_important_df['Importance'] > threshold_important
mask_split = feature_important_df['Num_Split'] > threshold_split

important_features = feature_important_df[mask_important & mask_split]['Feature'].values
# important_features = feature_important_df[feature_important_df['Importance'] > threshold_important]['Feature'].values

In [62]:
feature_names_lightgbm = []
for i, feature in enumerate(feature_names):
    
    feature_names_lightgbm.append(re.sub(r'[^\w]','_', feature))
feature_map = np.isin(feature_names_lightgbm, important_features)

In [63]:
len(important_features)

843

# Modelling

In [64]:
from sklearn.metrics import make_scorer, roc_auc_score
def gini_coefficient(y_true, y_pred):
    """
    Calculate the Gini coefficient using predictions and true labels.
    
    Parameters:
    y_true (array-like): True binary labels.
    y_pred (array-like): Predicted probabilities.
    
    Returns:
    float: Gini coefficient.
    """
    auc = roc_auc_score(y_true, y_pred)  # AUC calculation
    return 2 * auc - 1  # Gini coefficient
gini_scorer = make_scorer(gini_coefficient, needs_proba=True)

In [65]:
logistic_model = LogisticRegression(random_state=42, max_iter=10000, class_weight='balanced', C = 0.001, tol = 2e-5, n_jobs=-1)

In [66]:
logistic_model.fit(X_resampled[:, feature_map], y_resampled)
y_pred = logistic_model.predict_proba(X_test[:, feature_map])

In [67]:
submission = pd.DataFrame({
    'SK_ID_CURR': test_id,
    'TARGET': y_pred[:, 1]
})


In [68]:
ground_truth = pd.read_csv('../temp/target.csv')
submission2 = submission.merge(ground_truth, on='SK_ID_CURR', how='left')
gini_coefficient(submission2['True_Target'], submission2['TARGET'])

0.5644894851432054

In [69]:
# submission.to_csv('../submission/log_knn_oversample_2.csv', index=False)

In [70]:
0.567574126923756

0.567574126923756